# Manual R-vs-Python validation for `v1.1.1-alpha`

This notebook is for interactive checking of the migrated Python package against the upstream R package.

What it does:
- loads the local Python package from `src/`
- runs the upstream R package through `tools/generate_r_parity_suite.R`
- compares table outputs against R for all migrated table/list functions
- renders plot outputs side by side with the R reference plus an image diff

Default upstream R checkout:
- `/tmp/peptidomicsR-v1.1.1-alpha`

If your checkout is elsewhere, set `PEPTIDOMICSR_UPSTREAM_DIR` before starting the notebook.


In [ ]:
%matplotlib inline

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import warnings

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from pandas.testing import assert_frame_equal


In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing pyproject.toml")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

FIXTURE_DIR = REPO_ROOT / "tests" / "fixtures" / "v1.1.1-alpha"
REFERENCE_DIR = REPO_ROOT / ".migration-artifacts" / "v1.1.1-alpha" / "reference_outputs"
UPSTREAM_R_DIR = Path(os.environ.get("PEPTIDOMICSR_UPSTREAM_DIR", "/tmp/peptidomicsR-v1.1.1-alpha"))
NOTEBOOK_TMP = Path(tempfile.gettempdir()) / "peptidomicspy_manual_validation_v1_1_1_alpha"
NOTEBOOK_TMP.mkdir(parents=True, exist_ok=True)
R_PARITY_DIR = NOTEBOOK_TMP / "r_parity_suite"
PY_PLOT_DIR = NOTEBOOK_TMP / "py_plots"
PY_PLOT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = json.loads((REFERENCE_DIR / "manifest.json").read_text(encoding="utf-8"))

print(f"Repo root: {REPO_ROOT}")
print(f"Fixture dir: {FIXTURE_DIR}")
print(f"Frozen reference dir: {REFERENCE_DIR}")
print(f"Upstream R dir: {UPSTREAM_R_DIR}")
print(f"Notebook temp dir: {NOTEBOOK_TMP}")


In [ ]:
from matplotlib.figure import Figure
from plotnine import ggplot

from peptidomicspy import (
    PeptidomicsResult,
    calculate_gravy,
    filterPeptides,
    load_example_protein_mapping,
    plot_cleavage_site,
    plot_count,
    plot_gravy_vs_intensity,
    plot_int,
    plot_length_distribution,
    plot_pep_align,
    plot_type_num,
    plot_volcano,
    processPeptides,
    ttestPeptides,
)

print("Imported the full public API successfully.")


In [ ]:
def run_r_parity_suite(force: bool = False) -> Path:
    if force and R_PARITY_DIR.exists():
        shutil.rmtree(R_PARITY_DIR)
    if R_PARITY_DIR.exists():
        return R_PARITY_DIR
    if not UPSTREAM_R_DIR.exists():
        raise FileNotFoundError(
            f"Upstream R checkout not found at {UPSTREAM_R_DIR}. "
            "Set PEPTIDOMICSR_UPSTREAM_DIR and restart the notebook."
        )
    env = os.environ.copy()
    env.setdefault("R_LIBS_USER", str(Path.home() / "R" / "x86_64-pc-linux-gnu-library" / "4.3"))
    subprocess.run(
        [
            "Rscript",
            "tools/generate_r_parity_suite.R",
            str(UPSTREAM_R_DIR),
            str(FIXTURE_DIR),
            str(R_PARITY_DIR),
        ],
        cwd=REPO_ROOT,
        env=env,
        check=True,
    )
    return R_PARITY_DIR


def read_reference_frame(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(REFERENCE_DIR / relative_path)


def read_r_parity_frame(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(R_PARITY_DIR / relative_path)


def normalize_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy(deep=True)
    for col in out.columns:
        if isinstance(out[col].dtype, pd.CategoricalDtype):
            out[col] = out[col].astype(str)
    return out


def assert_frame_close(actual: pd.DataFrame, reference: pd.DataFrame) -> None:
    actual_norm = normalize_frame(actual)
    reference_norm = normalize_frame(reference)
    assert list(actual_norm.columns) == list(reference_norm.columns)

    sort_cols = list(reference_norm.columns)
    actual_norm = actual_norm.sort_values(by=sort_cols, kind="mergesort").reset_index(drop=True)
    reference_norm = reference_norm.sort_values(by=sort_cols, kind="mergesort").reset_index(drop=True)

    numeric_cols = [
        col
        for col in reference_norm.columns
        if pd.api.types.is_numeric_dtype(reference_norm[col])
        or pd.api.types.is_numeric_dtype(actual_norm[col])
    ]
    other_cols = [col for col in reference_norm.columns if col not in numeric_cols]

    if other_cols:
        assert_frame_equal(
            actual_norm[other_cols],
            reference_norm[other_cols],
            check_dtype=False,
            check_like=False,
        )
    for col in numeric_cols:
        left = pd.to_numeric(actual_norm[col], errors="coerce").to_numpy()
        right = pd.to_numeric(reference_norm[col], errors="coerce").to_numpy()
        np.testing.assert_allclose(left, right, rtol=1e-7, atol=1e-9, equal_nan=True)


def get_plot_label(plot, key: str = "color"):
    labels = getattr(plot, "labels", None)
    if labels is None:
        return None
    for candidate in (key, "colour" if key == "color" else key):
        if hasattr(labels, candidate):
            value = getattr(labels, candidate)
            if value is not None:
                return value
        if hasattr(labels, "get"):
            value = labels.get(candidate)
            if value is not None:
                return value
    return None


def load_rgb_image(path: Path) -> np.ndarray:
    arr = mpimg.imread(path).astype(float)
    if arr.max() > 1:
        arr = arr / 255.0
    if arr.shape[-1] == 4:
        alpha = arr[..., 3:4]
        arr = arr[..., :3] * alpha + (1 - alpha)
    return arr[..., :3]


def resize_nn(image: np.ndarray, height: int, width: int) -> np.ndarray:
    ys = np.linspace(0, image.shape[0] - 1, height).round().astype(int)
    xs = np.linspace(0, image.shape[1] - 1, width).round().astype(int)
    return image[np.ix_(ys, xs)]


def image_mae(path_a: Path, path_b: Path) -> float:
    image_a = load_rgb_image(path_a)
    image_b = load_rgb_image(path_b)
    height = max(image_a.shape[0], image_b.shape[0])
    width = max(image_a.shape[1], image_b.shape[1])
    image_a = resize_nn(image_a, height, width)
    image_b = resize_nn(image_b, height, width)
    return float(np.abs(image_a - image_b).mean())


def save_plot_object(obj, path: Path, width: float, height: float) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(obj, ggplot):
        obj.save(path, width=width, height=height, dpi=150, verbose=False)
        return path
    if isinstance(obj, Figure):
        obj.set_size_inches(width, height)
        obj.savefig(path, dpi=150)
        return path
    raise TypeError(f"Unsupported plot object type: {type(obj)!r}")


def show_plot_comparison(case_name: str, obj, width: float, height: float, threshold: float) -> None:
    py_path = save_plot_object(obj, PY_PLOT_DIR / f"{case_name}.png", width, height)
    r_path = R_PARITY_DIR / "plots" / f"{case_name}.png"
    mae = image_mae(py_path, r_path)

    py_image = load_rgb_image(py_path)
    r_image = load_rgb_image(r_path)
    height_px = max(py_image.shape[0], r_image.shape[0])
    width_px = max(py_image.shape[1], r_image.shape[1])
    py_image = resize_nn(py_image, height_px, width_px)
    r_image = resize_nn(r_image, height_px, width_px)
    diff = np.abs(py_image - r_image)

    status = "PASS" if mae <= threshold else "CHECK"
    display(Markdown(f"**{case_name}**: `{status}` with image MAE `{mae:.4f}` against threshold `{threshold:.4f}`"))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].imshow(r_image)
    axes[0].set_title("R reference")
    axes[1].imshow(py_image)
    axes[1].set_title("Python")
    axes[2].imshow(np.clip(diff * 4, 0, 1))
    axes[2].set_title("Absolute diff x4")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)


## 1. Build the live R parity references

Run the next cell first. It regenerates the comparison suite from the upstream R package into a temporary directory.

If you changed the R package checkout or want a clean rebuild, re-run with `force=True`.


In [ ]:
run_r_parity_suite(force=False)
print(f"Live R parity suite: {R_PARITY_DIR}")


## 2. Packaged example mapping and GRAVY helper

These checks are quick sanity checks for the packaged dataset helper and `calculate_gravy()`.


In [ ]:
mapping = load_example_protein_mapping()
assert isinstance(mapping, pd.DataFrame)
assert not mapping.empty
assert {"Leading.razor.protein", "Protein.name", "Protein.group", "Protein.seq"}.issubset(mapping.columns)

assert calculate_gravy("A") == 1.8
assert np.isnan(calculate_gravy("AX"))

display(mapping.head())
display(
    pd.DataFrame(
        {
            "sequence": ["A", "AAIDEASKKLNAQ", "AAGGPGAPADPGRPT", "GGGGGG"],
            "gravy_score": [
                calculate_gravy("A"),
                calculate_gravy("AAIDEASKKLNAQ"),
                calculate_gravy("AAGGPGAPADPGRPT"),
                calculate_gravy("GGGGGG"),
            ],
        }
    )
)


## 3. `processPeptides()`

This compares the Python result to the frozen R reference tables committed in this repo.


In [ ]:
EXPECTED_COUNTS = {
    "dt_peptides": 3155,
    "dt_peptides_int": 7671,
    "dt_peptides_int_reps": 21825,
    "dt_peptides_int_ttest": 18930,
    "dt_peptides_typenum": 512,
    "dt_peptides_typenum_reps": 1512,
}

PROCESS_REFERENCE_TABLES = {
    "dt_int_col": "processPeptides/dt_int_col.csv.gz",
    "dt_peptides": "processPeptides/dt_peptides.csv.gz",
    "dt_peptides_int": "processPeptides/dt_peptides_int.csv.gz",
    "dt_peptides_int_reps": "processPeptides/dt_peptides_int_reps.csv.gz",
    "dt_peptides_int_ttest": "processPeptides/dt_peptides_int_ttest.csv.gz",
    "dt_peptides_typenum": "processPeptides/dt_peptides_typenum.csv.gz",
    "dt_peptides_typenum_reps": "processPeptides/dt_peptides_typenum_reps.csv.gz",
}

result = processPeptides(
    peptides_file=FIXTURE_DIR / "Yogurtexample_QR188-205.csv",
    intensity_columns_file=FIXTURE_DIR / "Intensity_columns.csv",
    protein_mapping_file=FIXTURE_DIR / "protein_mapping.csv",
)

assert isinstance(result, PeptidomicsResult)
assert result.grp_cols == ["Digest.stage", "Yogurt"]

observed_counts = {
    "dt_peptides": len(result.dt_peptides),
    "dt_peptides_int": len(result.dt_peptides_int),
    "dt_peptides_int_reps": len(result.dt_peptides_int_reps),
    "dt_peptides_int_ttest": len(result.dt_peptides_int_ttest),
    "dt_peptides_typenum": len(result.dt_peptides_typenum),
    "dt_peptides_typenum_reps": len(result.dt_peptides_typenum_reps),
}
assert observed_counts == EXPECTED_COUNTS

for attr, relpath in PROCESS_REFERENCE_TABLES.items():
    assert_frame_close(getattr(result, attr), read_reference_frame(relpath))

print("processPeptides() matches the frozen R reference outputs.")
display(pd.DataFrame([observed_counts]))
display(result.dt_peptides.head(3))


## 4. `filterPeptides()`

This section compares all tracked filter modes against live R outputs:
- exact sequence filter
- regex filter
- combined regex plus grouping filter
- union of `seqs` and `seq_pattern`


In [ ]:
filter_cases = {
    "exact": {
        "kwargs": {"seqs": ["AAGGPGAPADPGRPT", "AAIDEASKKLNAQ"]},
        "tables": ["dt_peptides", "dt_peptides_int", "dt_peptides_int_reps", "dt_peptides_typenum", "dt_peptides_typenum_reps", "dt_int_col"],
    },
    "regex": {
        "kwargs": {"seq_pattern": "^AA"},
        "tables": ["dt_peptides", "dt_peptides_int", "dt_peptides_int_reps", "dt_peptides_typenum", "dt_peptides_typenum_reps", "dt_int_col"],
    },
    "grouped": {
        "kwargs": {"seq_pattern": "DQAM", "filter_params": {"Yogurt": "Y1", "Digest.stage": "G120"}},
        "tables": ["dt_peptides", "dt_peptides_int", "dt_peptides_int_reps", "dt_peptides_typenum", "dt_peptides_typenum_reps", "dt_int_col"],
    },
    "union": {
        "kwargs": {"seqs": ["AAGGPGAPADPGRPT"], "seq_pattern": "^AAI"},
        "tables": ["dt_peptides", "dt_peptides_int", "dt_peptides_int_reps", "dt_peptides_typenum", "dt_peptides_typenum_reps", "dt_int_col"],
    },
}

filter_summary = []
for case_name, spec in filter_cases.items():
    actual = filterPeptides(result, **spec["kwargs"])
    for table_name in spec["tables"]:
        reference = read_r_parity_frame(f"tables/filter/{case_name}/{table_name}.csv.gz")
        assert_frame_close(getattr(actual, table_name), reference)
    filter_summary.append(
        {
            "case": case_name,
            "dt_peptides_rows": len(actual.dt_peptides),
            "dt_peptides_int_rows": len(actual.dt_peptides_int),
            "dt_ttest_rows": len(actual.dt_peptides_int_ttest),
        }
    )

# extra sanity check for the downstream t-test table after filtering
filtered_grouped = filterPeptides(
    result,
    seq_pattern="DQAM",
    filter_params={"Yogurt": "Y1", "Digest.stage": "G120"},
)
assert set(filtered_grouped.dt_peptides_int_ttest["Yogurt"].astype(str).unique()) == {"Y1"}
assert set(filtered_grouped.dt_peptides_int_ttest["Digest.stage"].astype(str).unique()) == {"G120"}

print("All filterPeptides() parity cases passed.")
display(pd.DataFrame(filter_summary))


## 5. `ttestPeptides()`

This section checks:
- plain mode against multiple selector styles
- pooled partial selectors
- equal-variance plain mode
- treat-mode agreement against the R reference


In [ ]:
def compare_ttest_core(actual: pd.DataFrame, reference: pd.DataFrame) -> dict[str, object]:
    merged = actual.merge(
        reference[["Sequence", "t", "df", "p.value", "log2FC", "p.adj", "sig"]],
        on="Sequence",
        suffixes=("_py", "_r"),
    )
    np.testing.assert_allclose(merged["t_py"], merged["t_r"], rtol=1e-7, atol=1e-9, equal_nan=True)
    np.testing.assert_allclose(merged["df_py"], merged["df_r"], rtol=1e-7, atol=1e-9, equal_nan=True)
    np.testing.assert_allclose(merged["p.value_py"], merged["p.value_r"], rtol=1e-7, atol=1e-9, equal_nan=True)
    np.testing.assert_allclose(merged["log2FC_py"], merged["log2FC_r"], rtol=1e-8, atol=1e-8, equal_nan=True)
    np.testing.assert_allclose(merged["p.adj_py"], merged["p.adj_r"], rtol=1e-7, atol=1e-9, equal_nan=True)
    sig_match = float((merged["sig_py"] == merged["sig_r"]).mean())
    return {"rows": len(actual), "sig_match": sig_match}


plain_cases = {
    "plain_main": {
        "kwargs": {"comparisons": [("G120_Y1", "I120_Y1")]},
        "result_key": "G120_Y1_vs_I120_Y1",
    },
    "plain_reordered": {
        "kwargs": {"comparisons": [("Y1_G120", "Y1_I120")]},
        "result_key": "Y1_G120_vs_Y1_I120",
    },
    "plain_explicit": {
        "kwargs": {"comparisons": [("Digest.stage=G120_Yogurt=Y1", "Digest.stage=I120_Yogurt=Y1")]},
        "result_key": "Digest.stage=G120_Yogurt=Y1_vs_Digest.stage=I120_Yogurt=Y1",
    },
    "plain_pooled": {
        "kwargs": {"comparisons": [("Y1", "Y2")]},
        "result_key": "Y1_vs_Y2",
    },
    "plain_equalvar": {
        "kwargs": {"comparisons": [("G120_Y1", "I120_Y1")], "equal_var": True},
        "result_key": "G120_Y1_vs_I120_Y1",
    },
}

plain_summary = []
for case_name, spec in plain_cases.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        actual = ttestPeptides(result, test_method="plain", **spec["kwargs"])[spec["result_key"]]
    reference = read_r_parity_frame(f"tables/ttest/{case_name}.csv.gz")
    metrics = compare_ttest_core(actual, reference)
    plain_summary.append({
        "case": case_name,
        "rows": metrics["rows"],
        "sig_match": metrics["sig_match"],
        "warning": str(caught[0].message).splitlines()[0] if caught else "",
    })

actual_treat = ttestPeptides(
    result,
    comparisons=[("G120_Y1", "I120_Y1")],
    test_method="treat",
)["G120_Y1_vs_I120_Y1"]
reference_treat = read_r_parity_frame("tables/ttest/treat_main.csv.gz")
reference_treat_key = "G120_Y1_vs_I120_Y1"
reference_treat_labels = list(reference_treat["Sequence"].head(2))
reference_treat_dict = {reference_treat_key: reference_treat}
merged_treat = actual_treat.merge(reference_treat[["Sequence", "log2FC", "sig"]], on="Sequence", suffixes=("_py", "_r"))
np.testing.assert_allclose(merged_treat["log2FC_py"], merged_treat["log2FC_r"], rtol=1e-8, atol=1e-8, equal_nan=True)
treat_sig_agreement = float((merged_treat["sig_py"] == merged_treat["sig_r"]).mean())
assert treat_sig_agreement >= 0.95

print("All ttestPeptides() parity checks passed.")
display(pd.DataFrame(plain_summary))
print(f"treat significance agreement: {treat_sig_agreement:.2%}")


## 6. Plot parity against live R renders

Each cell below:
- renders the R reference image from the parity suite
- renders the Python plot for the same case
- shows R, Python, and an absolute-difference image
- reports the image mean absolute error (MAE)

Interpretation:
- `PASS` means the image MAE is within the branch threshold for that case
- `CHECK` means the plot still deserves manual inspection even if the code runs


In [ ]:
plot_cases = {
    "plot_int_mean_named": {
        "builder": lambda: plot_int(result, type="mean", x_var="Yogurt", filter_params={"Digest.stage": "G120"}, color_by="Protein.name"),
        "width": 8,
        "height": 5,
        "threshold": 0.12,
    },
    "plot_int_reps_default": {
        "builder": lambda: plot_int(result, type="reps"),
        "width": 8,
        "height": 5,
        "threshold": 0.07,
    },
    "plot_type_num_reps": {
        "builder": lambda: plot_type_num(result, type="reps", x_var="Yogurt", facet_rows="Digest.stage", color_by="Protein.group"),
        "width": 8,
        "height": 5,
        "threshold": 0.08,
    },
    "plot_count_mean_none": {
        "builder": lambda: plot_count(result, type="mean", x_var="Yogurt", color_by="none"),
        "width": 8,
        "height": 5,
        "threshold": 0.03,
    },
    "plot_length_bar": {
        "builder": lambda: plot_length_distribution(result, metric="intensity", filter_params={"Digest.stage": "G120"}, facet_rows="Yogurt"),
        "width": 8,
        "height": 5,
        "threshold": 0.08,
    },
    "plot_length_density": {
        "builder": lambda: plot_length_distribution(result, type="reps", metric="type_num", plot_mode="density", facet_rows="Digest.stage"),
        "width": 8,
        "height": 5,
        "threshold": 0.05,
    },
    "plot_gravy_scatter": {
        "builder": lambda: plot_gravy_vs_intensity(result),
        "width": 8,
        "height": 5,
        "threshold": 0.08,
    },
    "plot_gravy_density": {
        "builder": lambda: plot_gravy_vs_intensity(result, type="reps", plot_mode="density", facet_rows="Digest.stage", filter_params={"Yogurt": "Y1"}),
        "width": 8,
        "height": 5,
        "threshold": 0.05,
    },
    "plot_volcano_plain": {
        "builder": lambda: plot_volcano(
            ttestPeptides(result, comparisons=[("G120_Y1", "I120_Y1")], test_method="plain"),
            comparisons=[("G120_Y1", "I120_Y1")],
            test_method="plain",
        )["G120_Y1_vs_I120_Y1"],
        "width": 8,
        "height": 5,
        "threshold": 0.05,
    },
    "plot_volcano_treat_annotated": {
        "builder": lambda: plot_volcano(
            reference_treat_dict,
            comparisons=[("G120_Y1", "I120_Y1")],
            test_method="treat",
            label_seqs=reference_treat_labels,
            highlight_seqs={"black": [reference_treat_labels[0]]},
        )[reference_treat_key],
        "width": 8,
        "height": 5,
        "threshold": 0.06,
    },
    "plot_pep_align_mean": {
        "builder": lambda: plot_pep_align(result, protein_name="P81265", protein_seq="A" * 650, auto_size=False),
        "width": 12,
        "height": 8,
        "threshold": 0.10,
    },
    "plot_cleavage_N": {
        "builder": lambda: plot_cleavage_site(result, terminal="N"),
        "width": 8,
        "height": 5,
        "threshold": 0.20,
    },
    "plot_cleavage_both_filtered": {
        "builder": lambda: plot_cleavage_site(result, terminal="both", measure="type_num", replicate_mode="reps", filter_params={"Yogurt": "Y1"}),
        "width": 12,
        "height": 5,
        "threshold": 0.23,
    },
}


### 6A. `plot_int()`, `plot_type_num()`, and `plot_count()`

Run this cell to inspect the stacked bar plots and the deprecated compatibility wrapper.


In [ ]:
for case_name in ["plot_int_mean_named", "plot_int_reps_default", "plot_type_num_reps", "plot_count_mean_none"]:
    spec = plot_cases[case_name]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        obj = spec["builder"]()
    show_plot_comparison(case_name, obj, spec["width"], spec["height"], spec["threshold"])


### 6B. `plot_length_distribution()` and `plot_gravy_vs_intensity()`

Run this cell to inspect both bar/scatter and density modes.


In [ ]:
for case_name in ["plot_length_bar", "plot_length_density", "plot_gravy_scatter", "plot_gravy_density"]:
    spec = plot_cases[case_name]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        obj = spec["builder"]()
    show_plot_comparison(case_name, obj, spec["width"], spec["height"], spec["threshold"])


### 6C. `plot_volcano()`

Run this cell to inspect the plain and annotated treat-style volcano plots.


In [ ]:
for case_name in ["plot_volcano_plain", "plot_volcano_treat_annotated"]:
    spec = plot_cases[case_name]
    obj = spec["builder"]()
    show_plot_comparison(case_name, obj, spec["width"], spec["height"], spec["threshold"])


### 6D. `plot_pep_align()` and `plot_cleavage_site()`

Run this cell to inspect the most layout-heavy plots.


In [ ]:
for case_name in ["plot_pep_align_mean", "plot_cleavage_N", "plot_cleavage_both_filtered"]:
    spec = plot_cases[case_name]
    obj = spec["builder"]()
    show_plot_comparison(case_name, obj, spec["width"], spec["height"], spec["threshold"])


## 7. Final status

If all `assert` statements passed and the image comparisons look acceptable for your use case, then this branch is behaving consistently with the current R parity suite.

For a fast command-line version of the same checks, run:

```bash
PYTHONPATH=src .venv/bin/python -m pytest tests/test_r_parity.py -q
```


In [ ]:
print("Notebook parity checks completed for v1.1.1-alpha.")
